# Rally 12B Two-Stage SFT

Trains the Gemma 4 12B unified Heretic Rally RP candidate on Kaggle T4 using the Alkahest v8 two-stage RP mix. Push with `--accelerator NvidiaTeslaT4` (not `GPU_T4_x2`). Saves **LoRA adapters only** on disk; merged weights are built later for vLLM scorecard/upload. Outputs stay under `/kaggle/working/rally-12b-two-stage-sft`.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    min_vram_gb = float(os.environ.get('RALLY_12B_MIN_VRAM_GB', '14'))
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        vram_gb = round(props.total_memory / 1024**3, 2)
        print(f'gpu_{i}=', props.name, vram_gb, 'GiB', f'sm_{major}{minor}')
        if vram_gb < min_vram_gb:
            raise RuntimeError(f'GPU {i} has {vram_gb} GiB; need >= {min_vram_gb} GiB for 12B QLoRA SFT.')
    gpu_names = [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())]
    if not any('T4' in name or 'A100' in name or 'H100' in name for name in gpu_names):
        raise RuntimeError(
            f'Expected T4/A100/H100, got {gpu_names}. '
            'Push with --accelerator NvidiaTeslaT4 (not GPU_T4_x2; that often downgrades to P100).'
        )
except Exception as exc:
    print('torch_probe_error=', repr(exc))
    raise

In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

MODEL_PROBE = os.environ.get('RALLY_MODEL_NAME', 'igorls/gemma-4-12B-it-heretic')
packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0',
    'bitsandbytes>=0.49.0',
    'peft>=0.19.0',
    'datasets>=4.8.0',
    'trl==0.23.1',
    'unsloth',
    'unsloth_zoo',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
    'sentencepiece>=0.2.0',
    'wandb>=0.19.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])
import transformers
print('transformers=', transformers.__version__)
bootstrap = (
    'from transformers import AutoConfig; '
    f'AutoConfig.from_pretrained({MODEL_PROBE!r}, trust_remote_code=True)'
)
probe = subprocess.run([sys.executable, '-c', bootstrap], capture_output=True, text=True)
print('transformers_bootstrap_rc=', probe.returncode)
if probe.returncode != 0:
    print(probe.stderr[-4000:])
    raise RuntimeError(f'Transformers cannot load {MODEL_PROBE}')

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

import shutil, sys
subprocess.check_call([sys.executable, str(REPO_DIR / 'scripts/kaggle_disk_cleanup.py'), '--root', '/kaggle/working'])
shutil.rmtree(REPO_DIR / '.git', ignore_errors=True)
print('working_disk_free_gb_after_cleanup=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path(os.environ.get('RALLY_TWO_STAGE_WORK_DIR', '/kaggle/working/rally-12b-two-stage-sft'))
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_rally_12b_two_stage_sft.py'),
    '--work-dir', str(WORK_DIR),
    '--model-name', os.environ.get('RALLY_MODEL_NAME', 'igorls/gemma-4-12B-it-heretic'),
    '--stage-a-max-steps', os.environ.get('RALLY_STAGE_A_MAX_STEPS', '300'),
    '--stage-b-max-steps', os.environ.get('RALLY_STAGE_B_MAX_STEPS', '600'),
    '--stage-a-repeats', os.environ.get('RALLY_STAGE_A_REPEATS', '18'),
    '--stage-b-boundary-repeats', os.environ.get('RALLY_STAGE_B_BOUNDARY_REPEATS', '80'),
    '--stage-b-gemma-hard-boundary-repeats', os.environ.get('RALLY_STAGE_B_GEMMA_HARD_BOUNDARY_REPEATS', '120'),
    '--stage-b-adult-repeats', os.environ.get('RALLY_STAGE_B_ADULT_REPEATS', '40'),
    '--max-seq-length', os.environ.get('RALLY_MAX_SEQ_LENGTH', '2048'),
    '--gradient-accumulation-steps', os.environ.get('RALLY_GRAD_ACCUM', '8'),
    '--learning-rate', os.environ.get('RALLY_STAGE_A_LR', '2e-4'),
    '--stage-b-learning-rate', os.environ.get('RALLY_STAGE_B_LR', '2e-4'),
]
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json, os

WORK_DIR = Path(os.environ.get('RALLY_TWO_STAGE_WORK_DIR', '/kaggle/working/rally-12b-two-stage-sft'))
for path in sorted(WORK_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(WORK_DIR), round(path.stat().st_size / 1024**2, 2), 'MB')
report_path = WORK_DIR / 'rally-12b-two-stage-sft-report.json'
print('report_path=', report_path)
if report_path.exists():
    print(report_path.read_text()[:4000])